In [3]:
pip install sklearn

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

ModuleNotFoundError: No module named 'sklearn'

In [6]:
train_transaction = pd.read_csv('train_transaction.csv')
train_identity = pd.read_csv('train_identity.csv')

data = train_transaction.merge(train_identity, how='left', on='TransactionID')


In [7]:
y = data['isFraud']
X = data.drop(['isFraud', 'TransactionID'], axis=1)


In [8]:
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(exclude=['object']).columns

for col in categorical_cols:
    X[col] = X[col].fillna('missing')
    X[col] = X[col].astype('category').cat.codes

for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())


In [9]:
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])


In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [11]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))


In [12]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
 model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['AUC']
)


In [14]:
early_stop = EarlyStopping(
    monitor='val_auc',
    patience=5,
    mode='max',
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=2048,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 15s 603ms/step - AUC: 0.6521 - loss: 0.7463 - val_AUC: 0.7802 - val_loss: 3.0383
Epoch 2/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.7869 - loss: 0.5956 - val_AUC: 0.7411 - val_loss: 0.1213
Epoch 3/30


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_auc` which is not available. Available metrics are: AUC,loss,val_AUC,val_loss
  current = self.get_monitor_value(logs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.7928 - loss: 0.5635 - val_AUC: 0.7802 - val_loss: 0.1244
Epoch 4/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.8298 - loss: 0.5284 - val_AUC: 0.7821 - val_loss: 0.1362
Epoch 5/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8225 - loss: 0.5156 - val_AUC: 0.7927 - val_loss: 0.2779
Epoch 6/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8483 - loss: 0.4949 - val_AUC: 0.7998 - val_loss: 0.2579
Epoch 7/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8563 - loss: 0.4782 - val_AUC: 0.7979 - val_loss: 0.2156
Epoch 8/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - AUC: 0.8633 - loss: 0.4645 - val_AUC: 0.8038 - val_loss: 0.1386
Epoch 9/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.8763 - loss: 0.4536 - val_AUC: 0.8126 - val_loss: 0.2398
Epoch 10/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - AUC: 0.8725 - loss: 0.4653 - val_AUC: 0.8148 - val_loss: 0.1860
Epoch 11/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.8826 - loss: 

In [15]:
val_preds = model.predict(X_val).ravel()
auc = roc_auc_score(y_val, val_preds)
print("Validation AUC:", auc)


182/182 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Validation AUC: 0.7694336283185841
